# EDTA-Co solve_trace diagnostic

Priority item #1 from the v3 handoff ("diagnose before fixing again"): inspect a non-converged EDTA trial's `solve_trace` (score, grad_norm, anchor/constraint terms over the T steps) to see whether it's oscillating, diverging, or genuinely plateaued at a bad score.

**Run this with the `jacobik-env` kernel selected** (the one with torch 2.12.1+cu130 / rdkit 2026.03.3 — same environment `reproduce_edta_results.py` targets). Run all cells top to bottom.

This notebook only *runs the solve and dumps the raw trace* to `edta_trace_dump.npz` / `edta_trace_meta.json` in this folder. The actual classification/plotting (oscillating vs diverging vs plateaued vs still-improving) happens offline in `analyze_edta_trace.py`, which is numpy-only and doesn't need this kernel — no need to run that part here.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# --- Force this notebook to import jacobik from the Jacobik v3 checkout ---
# A previous run resolved `import jacobik` to a DIFFERENT project
# (Jacobik v2\src\jacobik) instead of this one, silently. This produced a
# solve_trace with a completely different schema and a much higher
# convergence rate than the handoff describes -- i.e. we were diagnosing
# the wrong code. Only Jacobik v3 is wanted here, so: pin sys.path to it
# explicitly, and hard-fail if `jacobik` doesn't resolve to it.
#
# IMPORTANT: if you've run any cell in this kernel before that already
# imported `jacobik`, RESTART THE KERNEL first -- Python caches imports,
# so sys.path.insert alone won't undo an import that already happened.
import sys
V3_SRC = r'C:\Users\alanc\sus folder\CCCSEF\Jacobik v3\src'
sys.path.insert(0, V3_SRC)

import jacobik
import jacobik.ik.ik_solver as _ik_solver_mod

resolved = os.path.normcase(os.path.abspath(jacobik.__file__))
expected_prefix = os.path.normcase(os.path.abspath(V3_SRC))
print('jacobik resolved to:', jacobik.__file__)
print('ik_solver resolved to:', _ik_solver_mod.__file__)

if not resolved.startswith(expected_prefix):
    raise RuntimeError(
        f'jacobik resolved to {jacobik.__file__!r}, which is NOT under '
        f'{V3_SRC!r}. This almost certainly means `jacobik` (or a submodule) '
        f'was already imported earlier in this kernel session from a '
        f'different location (e.g. Jacobik v2) before this cell ran. '
        f'RESTART THE KERNEL and re-run this notebook from the top -- '
        f'do not just re-run this cell.'
    )
print('OK -- jacobik is resolving to the Jacobik v3 checkout.')

from reproduce_edta_results import check_environment
check_environment()

## Run parameters
Edit these and re-run the cells below if you want a different sweep (e.g. `STEPS = 400` to test the "just needs more steps" hypothesis).

In [ ]:
CONSTRAINTS_DB = 'constraints.json'
STEPS = 200        # n_ik_steps (T)
BEAM = 32          # beam_width (B)
SEED = 42
LR = 0.1
OUT_NPZ = 'edta_trace_dump.npz'
OUT_META = 'edta_trace_meta.json'

In [ ]:
from reproduce_edta_results import reproduce

result = reproduce(
    constraints_db_path=CONSTRAINTS_DB,
    seed=SEED,
    beam_width=BEAM,
    n_ik_steps=STEPS,
    lr=LR,
)
result.summary()

## Dump the raw trace
Saves the full `(T, B)` `solve_trace` arrays for *every* trial in the beam (not just kept conformers) plus per-conformer metadata, so the offline analysis can look at any trial by `trial_idx`.

In [ ]:
import json
import numpy as np

trace = result.solve_trace
assert trace, 'result.solve_trace is empty -- did the solve run any steps?'

conformers = result.conformers
conv_flags = np.array([c.converged for c in conformers], dtype=bool)
trial_idxs = np.array([c.trial_idx for c in conformers], dtype=np.int64)
scores = np.array([c.score for c in conformers], dtype=np.float64)

print(f'{int(conv_flags.sum())}/{len(conformers)} kept conformers converged (plateau criterion)')
if len(scores):
    print(f'score range among kept conformers: {scores.min():.4f} .. {scores.max():.4f}')
print('solve_trace arrays (T, B):', {k: v.shape for k, v in trace.items()})

In [ ]:
np.savez_compressed(
    OUT_NPZ,
    **trace,
    kept_trial_idx=trial_idxs,
    kept_score=scores,
    kept_converged=conv_flags,
)

meta = {
    'steps': STEPS,
    'beam_width': BEAM,
    'seed': SEED,
    'lr': LR,
    'n_kept': len(conformers),
    'n_converged': int(conv_flags.sum()),
    'best_score': float(scores.max()) if len(scores) else None,
    'trace_arrays': sorted(trace.keys()),
    'kept_conformers': [
        {'rank': c.rank, 'trial_idx': int(c.trial_idx), 'score': float(c.score), 'converged': bool(c.converged)}
        for c in conformers
    ],
}
with open(OUT_META, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Wrote {OUT_NPZ} and {OUT_META} -- done, nothing else to run here.')